> Projeto Desenvolve <br>
Programação Intermediária com Python <br>
Profa. Camila Laranjeira (mila@projetodesenvolve.com.br) <br>

# 3.11 - Data Model

## Exercícios

#### Q1. `dataclass`
Exercício adaptado de [codechalleng.es/bites/154/](https://codechalleng.es/bites/154/) e [codechalleng.es/bites/320/](https://codechalleng.es/bites/320/).

Neste desafio, você deve escrever uma `dataclass` chamada `Bite` que gerencia 3 atributos: `number`, `title` e `level`. Seus tipos são:
* `number` - `int`, 
* `title` - `str`, 
* `level` -  classe `Enum` chamada `BiteLevel` com os atributos `Beginner`, `Intermediate`, `Advanced`. 

Exemplo de dado: `{'number': 154, 'title': 'Escreva uma dataclass', 'level': BiteLevel.Intermediate}`

As características dessa classe são:
* O atributo`level` tem um valor padrão `BiteLevel.Beginner`
* Uma coleção de objetos `Bite` tem que ser ordenável somente pelo atributo `number`
* Implemente o método especial `__str__` para imprimir o Bite na forma `f'{number} - {title} ({level})'`

Teste sua classe executando o seguinte código:
```python
bites = []
bites.append(Bite(154, 'Escreva uma dataclass', 'Intermediate'))
bites.append(Bite(1, 'Some n valores'))
bites.append(Bite(37, 'Reescreva um loop com recursão', 'Intermediate'))

for b in bites.sort(): print(b)
# Ordem esperada na saída:
# 1 - Some n valores (Beginner)
# 37 - Reescreva um loop com recursão (Intermediate)
# 154 - Escreva uma dataclass (Intermediate)
```

In [ ]:
from dataclasses import dataclass, field
from enum import Enum

# Classe Enum para definir os níveis permitidos
class BiteLevel(Enum):
    Beginner = "Beginner"
    Intermediate = "Intermediate"
    Advanced = "Advanced"

# A dataclass com order=True permite ordenação (sort)
@dataclass(order=True)
class Bite:
    number: int
    # compare=False garante que a ordenação ocorra apenas pelo 'number'
    title: str = field(compare=False) 
    level: BiteLevel = field(default=BiteLevel.Beginner, compare=False)

    def __str__(self):
        # Acessamos .value para imprimir a string do nível (ex: "Beginner")
        return f'{self.number} - {self.title} ({self.level.value})'

# Testando a classe
bites = []
bites.append(Bite(154, 'Escreva uma dataclass', BiteLevel.Intermediate))
bites.append(Bite(1, 'Some n valores'))
bites.append(Bite(37, 'Reescreva um loop com recursão', BiteLevel.Intermediate))

bites.sort()
for b in bites: 
    print(b)

#### Q2. `Pydantic`
> Adaptada desse [tutorial de Pydantic](https://github.com/adonath/scipy-2023-pydantic-tutorial/tree/main) criado por [Axel Donath](https://github.com/adonath) e [Nick Langellier](https://github.com/nlangellier).

Observe a seguinte lista de observações da previsão do tempo em Murmansk, Russia.
```python
data_samples = [
    {
        "date": "2023-05-20",
        "temperature": 62.2,
        "isCelsius": False,
        "airQualityIndex": "24",
        "sunriseTime": "01:26",
        "sunsetTime": "00:00",
    },
    {
        "date": "2023-05-21",
        "temperature": "64.4",
        "isCelsius": "not true",
        "airQualityIndex": 23,
        "sunriseTime": "01:10",
        "sunsetTime": "00:16",
    },
    {
        "date": "2023-05-22",
        "temperature": 14.4,
        "airQualityIndex": 21,
    },
]
```

Escreva um script que calcule e imprima a temperatura média (em Celsius) em Murmansk para as datas fornecidas. Em seu script, você deve incluir um modelo Pydantic que registre com sucesso todos os elementos dados. Note que:

* Algumas amostras estão faltando dados. Você deve decidir quando o atributo pode ter um valor padrão ou quando definí-lo como opcional (`typing.Optional`). 
* Você precisará implementar pelo menos um validador de campo para transformar atributos. Dica: teste primeiro quais vão falhar :)



In [ ]:
from pydantic import BaseModel, field_validator
from typing import Optional
from datetime import date, time

data_samples = [
    {
        "date": "2023-05-20",
        "temperature": 62.2,
        "isCelsius": False,
        "airQualityIndex": "24",
        "sunriseTime": "01:26",
        "sunsetTime": "00:00",
    },
    {
        "date": "2023-05-21",
        "temperature": "64.4",
        "isCelsius": "not true",
        "airQualityIndex": 23,
        "sunriseTime": "01:10",
        "sunsetTime": "00:16",
    },
    {
        "date": "2023-05-22",
        "temperature": 14.4,
        "airQualityIndex": 21,
    },
]

class WeatherObservation(BaseModel):
    date: date
    temperature: float
    isCelsius: bool = True # Padrão assumido caso não seja fornecido
    airQualityIndex: int
    sunriseTime: Optional[time] = None
    sunsetTime: Optional[time] = None

    @field_validator('isCelsius', mode='before')
    @classmethod
    def parse_is_celsius(cls, v):
        if isinstance(v, str):
            # Limpa strings customizadas que representam False
            if v.lower() in ('false', 'not true', 'f', 'no'):
                return False
            # Limpa strings customizadas que representam True
            if v.lower() in ('true', 't', 'yes'):
                return True
        return v

# Script para calcular a temperatura média em Celsius
celsius_temps = []

for sample in data_samples:
    obs = WeatherObservation(**sample)
    
    temp = obs.temperature
    if not obs.isCelsius:
        # Conversão de Fahrenheit para Celsius
        temp = (temp - 32) * 5.0 / 9.0
        
    celsius_temps.append(temp)

avg_temp = sum(celsius_temps) / len(celsius_temps)
print(f"A temperatura média em Murmansk é: {avg_temp:.2f} °C")

#### Q3
> Adaptada desse [tutorial de Pydantic](https://github.com/adonath/scipy-2023-pydantic-tutorial/tree/main) criado por [Axel Donath](https://github.com/adonath) e [Nick Langellier](https://github.com/nlangellier).

Na célula a seguir, coletamos dados reais de uma das principais APIs de previsão do tempo, [open-meteo](https://open-meteo.com/en/docs). Não se preocupe em entender esse código, o mais importante é entender o resultado que ele retorna, ilustrado a seguir para uma coleta da temperatura dos últimos 15 dias em Itabira -MG. Caso deseje alterar a cidade de coleta, basta alimentar a latitude e longitude desejada, como nas opções a seguir.
* Itabira: `'latitude': -19.656655787605846, 'longitude': -43.228922960534476`
* Bom Despacho: `'latitude': -19.726308457732443, 'longitude': -45.27462803349767`

```python
{
  "latitude": -19.5,
  "longitude": -43.375,
  "generationtime_ms": 0.01800060272216797,
  "utc_offset_seconds": 0,
  "timezone": "GMT",
  "timezone_abbreviation": "GMT",
  "elevation": 2.0,
  "hourly_units": {
    "time": "iso8601",
    "temperature_2m": "\u00b0C"
  },
  "hourly": {
    "time": [
      "2024-07-19T00:00",
      "2024-07-19T01:00",
      "2024-07-19T02:00",
      ...
    ],
    "temperature_2m": [
      21.9,
      20.9,
      20.0,
      ... 
    ]
  }
}
```

Você deve escrever um modelo Pydantic `OpenMeteo` que receba diretamente a resposta dessa API, através do comando:
```python
dados = OpenMeteo(**response)
``` 

Para comportar a estrutura hierárquica desse dicionário (é um dicionário com alguns dicionários internos), você deve criar uma classe Pydantic para cada dicionário interno (`HourlyUnits` e `Hourly`), com seus respectivos atributos. Essas classes serão atributos da classe principal `OpenMeteo`, que terá também os outros atributos da resposta (`latitude`, `longitude`, etc.).



In [ ]:
from pydantic import BaseModel
from typing import List

# Submodelo para o dicionário "hourly_units"
class HourlyUnits(BaseModel):
    time: str
    temperature_2m: str

# Submodelo para o dicionário "hourly"
class Hourly(BaseModel):
    time: List[str]
    temperature_2m: List[float]

# Modelo principal agregando a resposta inteira
class OpenMeteo(BaseModel):
    latitude: float
    longitude: float
    generationtime_ms: float
    utc_offset_seconds: int
    timezone: str
    timezone_abbreviation: str
    elevation: float
    hourly_units: HourlyUnits
    hourly: Hourly

# Validando os dados da API
dados = OpenMeteo(**response)
print(f"Modelo processado com sucesso! Foram carregadas {len(dados.hourly.time)} medições.")

In [ ]:
#### Escreva aqui seus modelos Pydantic

#### Q4. 

Com os dados carregados na questão anterior plote um gráfico de linha, com a biblioteca de sua preferência, onde o eixo `x` são os timestamps (data e hora) e o eixo `y` é a temperatura medida.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Convertendo as listas aninhadas do nosso modelo em um DataFrame
df_clima = pd.DataFrame({
    'Tempo': pd.to_datetime(dados.hourly.time),
    'Temperatura': dados.hourly.temperature_2m
})

# Configuração da plotagem
plt.figure(figsize=(14, 5))
plt.plot(df_clima['Tempo'], df_clima['Temperatura'], color='#e74c3c', linewidth=2)

# Customização visual
plt.title('Previsão de Temperatura (15 dias)', fontsize=14, weight='bold')
plt.xlabel('Data e Hora', fontsize=12)
plt.ylabel(f"Temperatura ({dados.hourly_units.temperature_2m})", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()